# B0 Feature Exploration Starter

Template for exploring bidless hand features during Arc B development.

**Usage:** Copy this notebook and rename with date prefix (e.g., `2026_01_28_trump_distributions.ipynb`)

In [ ]:
# Auto-reload for development
%load_ext autoreload
%autoreload 2

In [ ]:
# Standard imports
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from bid_euchre.features.hand_eval import get_hand_features

# Project imports
from bid_euchre.sim.deals import generate_deal

# Configure plots
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

## Generate Sample Hands

Use deterministic seeding for reproducibility.

In [ ]:
# Generate some sample hands
SEED = 42
N_DEALS = 100

hands_data = []
for deal_id in range(N_DEALS):
    hands = generate_deal(SEED, deal_id)
    for seat in range(4):
        hand = hands[seat]
        # Get features for hearts trump
        features = get_hand_features(hand, 'suit', 'H')
        hands_data.append({
            'deal_id': deal_id,
            'seat': seat,
            'cards': [f"{c.rank}{c.suit}" for c in hand],
            **features
        })

df = pd.DataFrame(hands_data)
print(f"Generated {len(df)} hand records")
df.head()

## Explore Feature Distributions

In [ ]:
# Plot trump count distribution
fig, ax = plt.subplots(figsize=(8, 5))
df['trump_count'].hist(ax=ax, bins=range(8), align='left', rwidth=0.8)
ax.set_xlabel('Trump Count')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of Trump Cards per Hand')
plt.tight_layout()

In [ ]:
# Feature correlation heatmap
numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in ['deal_id', 'seat']]

fig, ax = plt.subplots(figsize=(10, 8))
corr = df[numeric_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Feature Correlations')
plt.tight_layout()

## Next Steps

- Load actual bidless dataset from `scripts/collect_bidless_dataset.py`
- Compare features across contract types (suit vs high vs low)
- Correlate features with simulation outcomes
- Build B0 value model (hand + contract -> expected tricks)